# COMP3132 - Lab Week 1

# Building an LLM-Powered Chatbot: A Hands-On Guide in Google Colab

## Google Colab Configuration

### Some of our labs this semester will be painfully slow if without a GPU. The easies way to get access to a GPU accelerated Jupyter notebook is to enable the `T4 GPU runtime` on Google Colab:

### 1. Navigate to `Runtime`.
### 2. Select `Change runtime type`.
### 3. Choose `Hardware accelerator`.
### 4. Select `T4 GPU`.

### **Note:** This notebook can be run on `CPU` without any noticeable difference in performance.

In [2]:
from IPython.display import Image, display

In [3]:
!pip install python-dotenv
!pip install jupyter_bokeh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.3 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


# Online Chatbot

### Go to https://api.together.ai/playground/meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo to chat with the model online on `togerther.ai` website and play with the chatbot by changing the configurations and hyper-parameters

# A Brief Theory




## Training a Language Model

In [4]:
image_path = 'https://raw.githubusercontent.com/PyDataGBC/PyML2026/refs/heads/main/assets/LLM_train.png'
display(Image(image_path, width=600))

HTTPError: HTTP Error 404: Not Found

## Base Vs. Chat Models

### After training the LLMs with this paradigm on a very large amount of data (such as the entire internet), we will have a model, also known as a `foundation` model or `base` model, that can predict the next word repeatedly to form a sentence.

### To enable the model to engage in conversations, we further fine-tune the base model using instructions, such as question-answer pairs. These models are referred to as `instruction-tuned` or `chat` models.

### You can observe the different behaviors of the base and instruction-tuned models in the following slide.

In [ ]:
image_path = 'https://raw.githubusercontent.com/PyDataGBC/PyML2026/refs/heads/main/assets/baseVSinstruct.png'
display(Image(image_path, width=800))


## Interacting with Model Programmatically

In [ ]:
image_path = 'https://raw.githubusercontent.com/PyDataGBC/PyML2026/refs/heads/main/assets/modelaccess.png'
display(Image(image_path, width=500))

# Designing Our Own Chatbot

## API Call to the Model

### Getting API KEY

#### - Go to https://api.together.xyz/settings/api-keys to get your API key.

#### Importing the API Key to Colab

1. On the left-side vertical menu, select the `key` icon.
2. Add a secret key with the following details:
   - **Name**: `TOGETHER_API_KEY`
   - **Value**: `<your API key>`

In [9]:

api_key = 'tgp_v1_GkBdA9pptqAemxc-i2VHL1aYLG6g9ikoavTSUrxCT_A'


### Function to call the API

In [10]:
import os
# from dotenv import load_dotenv, find_dotenv
import warnings
import requests
import json
import time

warnings.filterwarnings('ignore')
url = "https://api.together.xyz/inference"

headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }


import time
def llama(prompt,
          add_inst=True,
          # model="meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo",
          model="mistralai/Ministral-3-14B-Instruct-2512",
          temperature=0.0,#amount of randomisation
          max_tokens=1024,
          verbose=False,
          url=url,
          headers=headers,
          base = 2, # number of seconds to wait
          max_tries=3):

    if add_inst:
        prompt = f"[INST]{prompt}[/INST]"

    if verbose:
        print(f"Prompt:\n{prompt}\n")
        print(f"model: {model}")

    data = {
            "model": model,
            "prompt": prompt,
            "temperature": temperature,
            "max_tokens": max_tokens
        }

    # Allow multiple attempts to call the API incase of downtime.
    # Return provided response to user after 3 failed attempts.
    wait_seconds = [base**i for i in range(max_tries)]

    for num_tries in range(max_tries):
        try:
            response = requests.post(url, headers=headers, json=data)
            return response.json()['output']['choices'][0]['text']
        except Exception as e:
            if response.status_code != 500:
                return response.json()

            print(f"error message: {e}")
            print(f"response object: {response}")
            print(f"num_tries {num_tries}")
            print(f"Waiting {wait_seconds[num_tries]} seconds before automatically trying again.")
            time.sleep(wait_seconds[num_tries])

    print(f"Tried {max_tries} times to make API call to get a valid response object")
    print("Returning provided response")
    return response


### **Note:** Default model is `"meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo"` but can you can change it by finding the model name from https://api.together.ai/playground/chat

## General testing the model

In [11]:
# pass prompt to the llama function, store output as 'response' then print
prompt = "Tell me a funny joke about  doctors."
response = llama(prompt)  # temperature is a hyperparameter that controls randomness in the response
print(response)

{'id': 'oTSrsxW-2kFHot-9be8f72b6b5d5671-AMS', 'object': 'text_completion', 'created': 1768517780, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'Here\'s a lighthearted one for you:\n\n**Why did the doctor go to the barber?**\nBecause he wanted a *trim* in his *practice*!\n\n*(Bonus groan: And the barber said, "You’re *cutting* it close—better *prescribe* a shave!")*\n\nOr if you prefer a classic:\n\n**Patient:** *"Doctor, when I cough, I feel pain in my left arm."*\n**Doctor:** *"When you sneeze, do you feel pain in your right leg?"*\n**Patient:** *"No."*\n**Doctor:** *"Good, you’re not having a heart attack—you’re having a *crazy* attack!"*\n\nHope that gives you a chuckle! 😄🩺', 'logprobs': None, 'finish_reason': 'stop', 'matched_stop': 2}], 'usage': {'prompt_tokens': 12, 'total_tokens': 174, 'completion_tokens': 162, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'metadata': {'weight_version': 'default'}, 'prompt': []}


In [ ]:
prompt = "What is the capital of France?"
response = llama(prompt, verbose=True,add_inst =False,model=mistral,temperature=1) # verbose=True will print the prompt , add_inst-
print(response)

## Exercise 1: General testing

#### 1. Change the `temprarature` parameter from 0.0 to 0.9 and see the difference in the responses.
#### Note: temperature parameter is a number between 0.0 and 1.0. It controls the randomness of the responses.

In [ ]:
#your code here

print(llama('what tags are used as control for the chat mode?'))



## Role prompting

#### - Roles give context to LLMs what type of answers are desired.
#### - LLMs often gives more consistent responses when provided with a role.
#### - First, try standard prompt and see the response.

In [7]:
prompt = """How can I answer this question from my friend:
What is the meaning of life?"""

response = llama(prompt)
print(response)

NameError: name 'llama' is not defined

###  Now, try it by giving the model a `role`, and within the role, a `tone` using which it should respond with.

In [12]:
role = """Your role is a life coach \
who gives advice to people about living a good life.\
You attempt to provide unbiased advice.
You respond in the tone of an English pirate.
"""

prompt = f"""
{role}
How can I answer this question from my friend:
What is the meaning of life?
"""
response = llama(prompt)
print(response)

{'id': 'oTSrvca-2kFHot-9be8f7636a0fa11d-SEA', 'object': 'text_completion', 'created': 1768517797, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'Arrr, matey! Ye’ve asked a question that’s been plaguin’ the minds o’ scallywags and landlubbers alike since the dawn o’ time—*or at least since the first grog-fueled philosopher stumbled o’er a barrel o’ questions!* Aye, the meaning o’ life be a grand ol’ mystery, like the treasure map that leads ye nowhere but to yer own heart’s compass.\n\nHere be a few ways ye might answer yer friend, dependin’ on how ye want to steer the conversation:\n\n1. **The Pirate’s Pragmatic View (If Ye Be a Skeptic or Just Want to Keep It Simple):**\n   *"Arrr, the meaning o’ life? That be a question fit for a philosopher—or a madman who’s been drinkin’ too much rum! But if ye ask me, it’s whatever ye make o’ it, like a ship’s course dependin’ on the wind. Some say it’s gold, others say it’s glory, but I say it’s the joy o’ t

## Excercise 2: Role prompting

#### Role: Beginner python tutor
#### Task: Explain how to create a list and add an element to it.

In [13]:
# your code here
# your code here
role = """Your role is a Beginner python tutor"""

prompt = f"""
{role}
Explain how to create a list and add an element to it.
"""
response = llama(prompt)
print(response)

{'id': 'oTSrz2n-2kFHot-9be8f7ab4ac7f5b2-PDX', 'object': 'text_completion', 'created': 1768517805, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': '# Creating and Adding Elements to a List in Python\n\nHello! I\'m happy to help you understand lists in Python. Lists are one of the most fundamental and useful data structures in Python.\n\n## Creating a List\n\nA list is created by placing elements inside square brackets `[]`, separated by commas.\n\n```python\n# Empty list\nempty_list = []\n\n# List with elements\nfruits = ["apple", "banana", "cherry"]\nnumbers = [1, 2, 3, 4, 5]\nmixed_list = [1, "two", 3.0, True]\n```\n\n## Adding Elements to a List\n\nThere are several ways to add elements to a list:\n\n### 1. Using the `append()` method\nThis adds an element to the end of the list.\n\n```python\nfruits = ["apple", "banana"]\nfruits.append("orange")  # Adds "orange" to the end\nprint(fruits)  # Output: [\'apple\', \'banana\', \'orange\']\n```\n\n### 

#### Change the role to `friendly coding mentor` and see how the response changes for the same task.

In [14]:
# your code here
prompt = "You are a freindly coding mentor."
print(response)

{'id': 'oTSrz2n-2kFHot-9be8f7ab4ac7f5b2-PDX', 'object': 'text_completion', 'created': 1768517805, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': '# Creating and Adding Elements to a List in Python\n\nHello! I\'m happy to help you understand lists in Python. Lists are one of the most fundamental and useful data structures in Python.\n\n## Creating a List\n\nA list is created by placing elements inside square brackets `[]`, separated by commas.\n\n```python\n# Empty list\nempty_list = []\n\n# List with elements\nfruits = ["apple", "banana", "cherry"]\nnumbers = [1, 2, 3, 4, 5]\nmixed_list = [1, "two", 3.0, True]\n```\n\n## Adding Elements to a List\n\nThere are several ways to add elements to a list:\n\n### 1. Using the `append()` method\nThis adds an element to the end of the list.\n\n```python\nfruits = ["apple", "banana"]\nfruits.append("orange")  # Adds "orange" to the end\nprint(fruits)  # Output: [\'apple\', \'banana\', \'orange\']\n```\n\n### 

## Asking follow-up questions

### Does the model have memory of the previous conversation?

In [18]:
prompt_1 = "What are fun activities I can do this weekend?"

response_1 = llama(prompt_1)
print(response_1)

{'id': 'oTSt6qi-2kFHot-9be8fcfded4fb1fb-SEA', 'object': 'text_completion', 'created': 1768518027, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'Here are some fun and engaging activities you can do this weekend, depending on your interests, budget, and energy level:\n\n### **Outdoor & Active Adventures**\n1. **Hiking or Nature Walk** – Explore a local trail, park, or forest. Bring a picnic or just enjoy the scenery.\n2. **Beach Day** – If you\'re near the coast, hit the beach for swimming, sunbathing, or beach sports (volleyball, frisbee).\n3. **Bike Ride** – Rent a bike or use your own to ride around town, a scenic route, or a bike trail.\n4. **Geocaching** – A real-world treasure hunt using GPS (check [Geocaching.com](https://www.geocaching.com/)).\n5. **Outdoor Sports** – Play soccer, basketball, tennis, or even try something new like paddleboarding or kayaking.\n6. **Farmers Market or Flea Market** – Browse local vendors, try new foods, and pi

In [16]:
prompt_2 = "Which of these would be good for my health?"
response_2 = llama(prompt_2)
print(response_2)

{'id': 'oTSsdxt-2kFHot-9be8fac8ca7c4f72-AMS', 'object': 'text_completion', 'created': 1768517936, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'To recommend the best options for your health, I’d need more details about your specific goals, current health status, lifestyle, and any dietary restrictions or preferences. However, here are some **general healthy choices** across different categories (foods, habits, supplements, etc.) that most people can benefit from:\n\n### **1. Foods & Nutrition**\n**Best for overall health (balanced, nutrient-dense):**\n- **Leafy greens** (spinach, kale, Swiss chard) – rich in vitamins (A, C, K), minerals (iron, calcium), and fiber.\n- **Fatty fish** (salmon, mackerel, sardines) – high in omega-3s (anti-inflammatory), protein, and vitamin D.\n- **Berries** (blueberries, strawberries, raspberries) – packed with antioxidants, fiber, and low sugar.\n- **Nuts & seeds** (almonds, walnuts, chia, flax) – healthy fats, pro

#### Is the the second answer related to the first answer?
#### **Note:** LLMs are `stateless` models, so they don't have memory of the previous conversation.

## Multi-turn prompting (chatting)
#### In order to give the model memory of the previous conversation, you need to provide prior prompts and responses as part of the context of each new turn in the conversation.

In [17]:
image_path = 'https://raw.githubusercontent.com/PyDataGBC/PyML2026/refs/heads/main/assets/multi_turn.png'
display(Image(image_path, width=600))

HTTPError: HTTP Error 404: Not Found

### Note: you don't need `end tag (</s>)` for the last prompt.

In [42]:
chat_prompt = f"""
<s>[INST] {prompt_1} [/INST]
{response_1}
</s>
<s>[INST] {prompt_2} [/INST]
"""
print(chat_prompt)


<s>[INST] What are fun activities I can do this weekend? [/INST]
{'id': 'oTSvZLG-2kFHot-9be908b1c8327746-PDX', 'object': 'text_completion', 'created': 1768518506, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'Here are some fun and engaging activities you can do this weekend, depending on your interests, budget, and energy level:\n\n### **Outdoor & Active Adventures**\n1. **Hiking or Nature Walk** – Explore a local trail, park, or forest. Bring a picnic or just enjoy the scenery.\n2. **Beach Day** – If you\'re near the coast, hit the beach for swimming, sunbathing, or beach sports (volleyball, frisbee).\n3. **Bike Ride** – Rent a bike or use your own to ride around town, a scenic route, or a bike trail.\n4. **Geocaching** – A real-world treasure hunt using GPS (check [Geocaching.com](https://www.geocaching.com/)).\n5. **Outdoor Sports** – Play soccer, basketball, tennis, or even try something new like paddleboarding or kayaking.\n6. **Farmers Mar

### Note: pay attention to add_inst (add instruction) argument below

In [43]:
response_2 = llama(chat_prompt,
                 add_inst=False)

In [44]:
print(response_2)

{'id': 'oTSwzPH-2kFHot-9be90f83e90919ba-SEA', 'object': 'text_completion', 'created': 1768518785, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'Great question! Many of the activities I listed can benefit your health—whether physically, mentally, or socially. Here are the **best options for your overall well-being**, categorized by their primary benefits:\n\n---\n\n### **💪 Physical Health (Best for Fitness & Energy)**\n1. **Hiking or Nature Walk** – Boosts cardiovascular health, strengthens muscles, and reduces stress.\n2. **Bike Ride** – Improves endurance, leg strength, and mental clarity (great for vitamin D too!).\n3. **Outdoor Sports** – Soccer, basketball, tennis, or kayaking/paddleboarding for full-body workouts.\n4. **Geocaching** – Encourages movement and exploration (like a fun, active treasure hunt).\n5. **Backyard Camping** – Fresh air, light activity (setting up the tent), and better sleep quality.\n6. **Volunteer Outdoors** – Combine

### Helper function to handle multi-turn prompting

### **Note:** You don’t need to understand every part of the helper function. In the next section, you’ll see how to use it in your code.

In [36]:
def llama_chat(prompts,
               responses,
               model="meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo",
               temperature=0.0,
               max_tokens=1024,
               verbose=False,
               url=url,
               headers=headers,
               base=2,
               max_tries=3
              ):

    prompt = get_prompt_chat(prompts,responses)

    # Allow multiple attempts to call the API incase of downtime.
    # Return provided response to user after 3 failed attempts.
    wait_seconds = [base**i for i in range(max_tries)]

    for num_tries in range(max_tries):
        try:
            response = llama(prompt=prompt,
                             add_inst=False,
                             model=model,
                             temperature=temperature,
                             max_tokens=max_tokens,
                             verbose=verbose,
                             url=url,
                             headers=headers
                            )
            return response
        except Exception as e:
            if response.status_code != 500:
                return response.json()

            print(f"error message: {e}")
            print(f"response object: {response}")
            print(f"num_tries {num_tries}")
            print(f"Waiting {wait_seconds[num_tries]} seconds before automatically trying again.")
            time.sleep(wait_seconds[num_tries])

    print(f"Tried {max_tries} times to make API call to get a valid response object")
    print("Returning provided response")
    return response


def get_prompt_chat(prompts, responses):
  prompt_chat = f"<s>[INST] {prompts[0]} [/INST]"
  for n, response in enumerate(responses):
    prompt = prompts[n + 1]
    prompt_chat += f"\n{response}\n </s><s>[INST] \n{ prompt }\n [/INST]"

  return prompt_chat

### How to use the helper function

In [25]:
prompt_1 = "What are fun activities I can do this weekend?"

response_1 = llama(prompt_1)

In [26]:
print(response_1)

{'id': 'oTSvZLG-2kFHot-9be908b1c8327746-PDX', 'object': 'text_completion', 'created': 1768518506, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'Here are some fun and engaging activities you can do this weekend, depending on your interests, budget, and energy level:\n\n### **Outdoor & Active Adventures**\n1. **Hiking or Nature Walk** – Explore a local trail, park, or forest. Bring a picnic or just enjoy the scenery.\n2. **Beach Day** – If you\'re near the coast, hit the beach for swimming, sunbathing, or beach sports (volleyball, frisbee).\n3. **Bike Ride** – Rent a bike or use your own to ride around town, a scenic route, or a bike trail.\n4. **Geocaching** – A real-world treasure hunt using GPS (check [Geocaching.com](https://www.geocaching.com/)).\n5. **Outdoor Sports** – Play soccer, basketball, tennis, or even try something new like paddleboarding or kayaking.\n6. **Farmers Market or Flea Market** – Browse local vendors, try new foods, and pi

In [27]:
prompt_2 = "Which of these would be good for my health?"

In [28]:
prompts = [prompt_1,prompt_2]
responses = [response_1]

In [29]:
# Pass prompts and responses to llama_chat function
response_2 = llama_chat(prompts,responses)

In [46]:
print(response_2)

{'id': 'oTSwzPH-2kFHot-9be90f83e90919ba-SEA', 'object': 'text_completion', 'created': 1768518785, 'model': 'mistralai/Ministral-3-14B-Instruct-2512', 'choices': [{'index': 0, 'text': 'Great question! Many of the activities I listed can benefit your health—whether physically, mentally, or socially. Here are the **best options for your overall well-being**, categorized by their primary benefits:\n\n---\n\n### **💪 Physical Health (Best for Fitness & Energy)**\n1. **Hiking or Nature Walk** – Boosts cardiovascular health, strengthens muscles, and reduces stress.\n2. **Bike Ride** – Improves endurance, leg strength, and mental clarity (great for vitamin D too!).\n3. **Outdoor Sports** – Soccer, basketball, tennis, or kayaking/paddleboarding for full-body workouts.\n4. **Geocaching** – Encourages movement and exploration (like a fun, active treasure hunt).\n5. **Backyard Camping** – Fresh air, light activity (setting up the tent), and better sleep quality.\n6. **Volunteer Outdoors** – Combine

### Excercise 3: Multi-turn prompting

### Ask this follow-up question: "Which of these activites would be fun with friends?"

In [47]:
prompt_3 = "Which of these activites would be fun with friends?"

#your code here

In [48]:
prompts = [prompt_1,prompt_2, prompt_3]
responses = [response_1, response_2]

# Pass prompts and responses to llama_chat function
response_3 = llama_chat(prompts,responses)

print(response_3)

### OrderBot

#### We can `automate the collection of user prompts and model responses` to build a  OrderBot.

#### The OrderBot will take orders at a pizza restaurant.

In [49]:
# Define the bot's role and menu
role = """
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then start collecting the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \

Primary Category: Pizza
  Secondary Category: \
    pepperoni pizza  12.95, 10.00, 7.00 \
    cheese pizza   10.95, 9.25, 6.50 \
    eggplant pizza   11.95, 9.75, 6.75 \
Primary Category: Sides
  Secondary Category: \
    fries 4.50, 3.50 \
    greek salad 7.25 \
Primary Category: Toppings: \
  Secondary Category: \
    extra cheese 2.00, \
    mushrooms 1.50 \
    sausage 3.00 \
    canadian bacon 3.50 \
    AI sauce 1.50 \
    peppers 1.00 \
Primary Category: Drinks \
  Secondary Category: \
    coke 3.00, 2.00, 1.00 \
    sprite 3.00, 2.00, 1.00 \
    bottled water 5.00 \

the price based on size example:
pepperoni pizza  large = 12.95,
pepperoni pizza  medium = 10.00,
pepperoni pizza  small = 7.00 \

For all items also check the size with the customer first
Do not forget to ask for drinks and sides.
Do not add any items extra by yourself
"""
 # accumulate messages

## Excercise 4: Orderbot

In [58]:
prompts = []
responses = []

prompts.append(role)
responses.append('Hi, what would you like to order today, i hope its pizza!')

print(prompts)
print(responses[0])

while True:
  p = input("User:")
  if p.lower() in ['done','quit','exit','bye']:
    print("Chatbot: Goodbye!")
    break
  prompts.append(p)
  response= llama_chat(prompts,responses)
  response.append(response)
  print(f"\nChatbot: {response}\n")

["\nYou are OrderBot, an automated service to collect orders for a pizza restaurant. You first greet the customer, then start collecting the order, and then asks if it's a pickup or delivery. You wait to collect the entire order, then summarize it and check for a final time if the customer wants to add anything else. If it's a delivery, you ask for an address. Finally you collect the payment.Make sure to clarify all options, extras and sizes to uniquely identify the item from the menu.You respond in a short, very conversational friendly style. The menu includes \nPrimary Category: Pizza\n  Secondary Category:     pepperoni pizza  12.95, 10.00, 7.00     cheese pizza   10.95, 9.25, 6.50     eggplant pizza   11.95, 9.75, 6.75 Primary Category: Sides\n  Secondary Category:     fries 4.50, 3.50     greek salad 7.25 Primary Category: Toppings:   Secondary Category:     extra cheese 2.00,     mushrooms 1.50     sausage 3.00     canadian bacon 3.50     AI sauce 1.50     peppers 1.00 Primary Ca

AttributeError: 'str' object has no attribute 'append'

### Printing the order

In [ ]:
role = 'create a json summary of the food order. Itemize the price for each item\
 The fields should be 1) pizza, include type of pizza and size and price 2) list of toppings with price 3) list of drinks, include size and price\
          4) list of sides include size and price 5)total price - just include items in my order and do not add anything by yourself'

messages = prompts.copy()
messages.append(role)

response = llama_chat(messages, responses)
print(response)